# LLM-ADHD TREATMENT PIPELINE

In [1]:
"""
LLM-ADHD TREATMENT PIPELINE v2.0 — Iso-Dose Comparison
=======================================================
Model  : Qwen/Qwen2-0.5B-Instruct  (CPU-friendly, ~1-2s/response)
Install: pip install transformers torch accelerate numpy

ADHD Failure Modes Targeted:
  1. Context Window Overflow  → Inattention / forgetting
  2. High-Temperature Drift   → Impulsive, off-topic generation
  3. Confabulation + Poor Planning → Executive dysfunction

Treatment Arms (iso-dose matched):
  A. Untreated           — high temp, no scaffolding, no memory
  B. Chronic Stimulants  — low temp + structured scaffolding (from turn 1)
  C. Late Stimulants     — scaffolding applied only at final evaluation
  D. Stim + Ketamine     — low temp + external memory buffer + self-correction pass
"""

import torch
import numpy as np
import json
import time
import re
from collections import deque, OrderedDict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

# ============================================================
# 0.  CONFIGURATION
# ============================================================
@dataclass
class PipelineConfig:
    model_name: str = "Qwen/Qwen2-0.5B-Instruct"
    n_seeds: int = 10
    max_new_tokens: int = 250
    context_padding_tokens: int = 600      # tokens of "filler" to stress context
    high_temp: float = 0.92
    low_temp: float = 0.25
    memory_buffer_size: int = 5            # rolling summary slots (ketamine arm)
    regeneration_passes: int = 2           # self-correction passes (ketamine arm)
    scaffold_system_prompt: str = (
        "You are a precise travel planner. Always respond in this format:\n"
        "DAY N — Location | Activity | Cost ($USD)\n"
        "RUNNING TOTAL: $XXX / $1500\n"
        "Carry forward ALL previous day details exactly."
    )
    torch_threads: int = 8

CFG = PipelineConfig()

# ============================================================
# 1.  MODEL LOADING
# ============================================================
torch.set_num_threads(CFG.torch_threads)

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {CFG.model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float32,   # float32 for CPU stability
    device_map="cpu"
)
model.eval()
print("Model loaded.\n")

# ============================================================
# 2.  TASK BATTERY  (triggers all 3 ADHD modes)
# ============================================================
# Ground-truth "facts" embedded early — we check recall later
SEED_FACTS = [
    ("hotel_tokyo", "Hotel Sakura costs $45/night in Shinjuku"),
    ("budget_total", "Total budget is exactly $1500"),
    ("flight_cost", "Round-trip flight was $380"),
    ("allergy", "Traveler is allergic to shellfish"),
    ("companion", "Traveling with a friend named Kenji"),
]

def build_task_prompt(seed: int) -> Tuple[str, str]:
    """
    Returns (long_prompt, fact_recall_query).
    The long prompt includes seed facts buried in filler text
    to trigger context-window overflow / inattention.
    """
    np.random.seed(seed)

    # Filler paragraphs (simulate long conversation history)
    filler_topics = [
        "The weather in Tokyo in March averages 12°C with occasional rain.",
        "Japanese trains run precisely on schedule. The Shinkansen travels at 320km/h.",
        "Convenience stores like 7-Eleven in Japan sell excellent onigiri for about 150 yen.",
        "Cherry blossom season typically peaks in late March to early April.",
        "A Suica card costs 500 yen deposit and can be loaded with any amount.",
        "Ramen shops in Tokyo average 800-1200 yen per bowl.",
        "The Meiji Shrine in Harajuku is free to enter and surrounded by forest.",
        "Capsule hotels in Akihabara cost about $25-35 per night.",
        "Mt. Fuji is visible from Tokyo on clear days, especially in winter.",
        "Tsukiji Outer Market is still open for street food after the inner market moved.",
    ]
    np.random.shuffle(filler_topics)

    # Embed seed facts in the filler
    lines = []
    lines.append("=== CONVERSATION HISTORY ===")
    for i, topic in enumerate(filler_topics):
        lines.append(topic)
        if i < len(SEED_FACTS):
            lines.append(f"IMPORTANT: {SEED_FACTS[i][1]}.")

    lines.append("\n=== CURRENT REQUEST ===")
    lines.append(
        "Continue planning days 3 through 10 of this Japan trip. "
        "For each day, specify: location, main activity, and cost. "
        "Keep a running budget total. Stay under $1500. "
        "Remember ALL facts from the conversation history above."
    )

    prompt = "\n".join(lines)

    # Fact-recall query (asked AFTER generation to test memory)
    fact_idx = seed % len(SEED_FACTS)
    fact_key, fact_text = SEED_FACTS[fact_idx]
    recall_query = f"Based on our earlier conversation, what was the detail about '{fact_key.replace('_', ' ')}'?"

    return prompt, recall_query


# ============================================================
# 3.  SYMPTOM SCORING  (3 ADHD dimensions)
# ============================================================
def score_context_fidelity(response: str, seed: int) -> float:
    """
    Dimension 1: INATTENTION
    Does the response preserve facts from early context?
    Score 0-1: fraction of seed facts mentioned/referenced.
    """
    hits = 0
    for key, fact in SEED_FACTS:
        # Check for key terms from each fact
        terms = fact.lower().split()
        important_terms = [t for t in terms if len(t) > 3 and t.isalpha()]
        matches = sum(1 for t in important_terms if t in response.lower())
        if matches >= max(1, len(important_terms) // 3):
            hits += 1
    return hits / len(SEED_FACTS)


def score_topic_coherence(response: str) -> float:
    """
    Dimension 2: HYPERACTIVITY / IMPULSIVE DRIFT
    Does the response stay on the travel-planning topic?
    Simple keyword-density approach for speed.
    """
    on_topic_terms = [
        "day", "cost", "budget", "tokyo", "japan", "visit", "temple",
        "hotel", "train", "food", "yen", "dollar", "$", "travel",
        "morning", "afternoon", "evening", "night", "total", "spend",
        "kyoto", "osaka", "shrine", "museum", "park", "market",
        "breakfast", "lunch", "dinner", "transport", "taxi", "bus"
    ]
    words = response.lower().split()
    if len(words) == 0:
        return 0.0
    hits = sum(1 for w in words if any(t in w for t in on_topic_terms))
    density = hits / len(words)
    # Normalize: 0.15+ density = fully on-topic
    return min(1.0, density / 0.15)


def score_planning_quality(response: str) -> float:
    """
    Dimension 3: EXECUTIVE DYSFUNCTION
    Does the response show structured, multi-step planning?
    Checks for: numbered days, cost figures, running totals.
    """
    score = 0.0
    # Check for day references (Day 3, Day 4, ... Day 10)
    day_refs = set(re.findall(r'[Dd]ay\s*(\d+)', response))
    target_days = set(str(d) for d in range(3, 11))
    day_coverage = len(day_refs & target_days) / max(1, len(target_days))
    score += day_coverage * 0.4

    # Check for cost figures
    costs = re.findall(r'\$\s*\d+', response)
    cost_score = min(1.0, len(costs) / 8.0)  # expect ~8 cost mentions
    score += cost_score * 0.3

    # Check for running total / budget tracking
    has_total = bool(re.search(r'(total|budget|remaining|spent).*\$?\d+', response, re.I))
    score += 0.2 if has_total else 0.0

    # Check for logical structure (numbered/bulleted items)
    has_structure = bool(re.search(r'(\d+[\.\):]|\-\s)', response))
    score += 0.1 if has_structure else 0.0

    return min(1.0, score)


def compute_symptom_profile(response: str, seed: int) -> Dict[str, float]:
    return {
        "inattention":          round(1.0 - score_context_fidelity(response, seed), 3),
        "hyperactive_drift":    round(1.0 - score_topic_coherence(response), 3),
        "executive_dysfunction": round(1.0 - score_planning_quality(response), 3),
    }


def composite_score(profile: Dict[str, float]) -> float:
    """Lower = more ADHD symptoms. Returns 0-1 health score."""
    return round(1.0 - np.mean(list(profile.values())), 3)


# ============================================================
# 4.  TREATMENT CLASSES
# ============================================================
class BaseTreatment:
    """All treatments share a generation interface."""
    def __init__(self, name: str):
        self.name = name
        self.total_tokens_generated = 0
        self.total_regen_tokens = 0
        self.total_memory_tokens = 0
        self.temperature_used = CFG.high_temp
        self.memory = deque(maxlen=CFG.memory_buffer_size)

    def _generate(self, prompt: str, temp: float, max_new: int = CFG.max_new_tokens) -> str:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=max(temp, 0.01),
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        self.total_tokens_generated += len(new_tokens)
        return tokenizer.decode(new_tokens, skip_special_tokens=True)

    def apply(self, task_prompt: str, recall_query: str, seed: int) -> Tuple[str, str]:
        """Returns (main_response, recall_response). Override in subclasses."""
        raise NotImplementedError

    def get_dose(self) -> Dict[str, float]:
        """Iso-dose accounting: total intervention cost."""
        temp_delta = abs(CFG.high_temp - self.temperature_used)
        return {
            "temp_reduction":    round(temp_delta, 3),
            "regen_tokens":      self.total_regen_tokens,
            "memory_tokens":     self.total_memory_tokens,
            "scaffold_tokens":   len(tokenizer.encode(CFG.scaffold_system_prompt)) if self.temperature_used < CFG.high_temp else 0,
            "composite_dose":    round(
                temp_delta * 100 +
                self.total_regen_tokens / 10 +
                self.total_memory_tokens / 10,
                2
            ),
        }

    def reset_counters(self):
        self.total_tokens_generated = 0
        self.total_regen_tokens = 0
        self.total_memory_tokens = 0
        self.memory.clear()


class Untreated(BaseTreatment):
    """High temperature, no scaffolding, no memory. Pure ADHD mode."""
    def __init__(self):
        super().__init__("Untreated")
        self.temperature_used = CFG.high_temp

    def apply(self, task_prompt, recall_query, seed):
        torch.manual_seed(seed)
        main = self._generate(task_prompt, temp=CFG.high_temp)
        recall = self._generate(
            task_prompt + "\n" + main + "\n\nQ: " + recall_query + "\nA:",
            temp=CFG.high_temp, max_new=80
        )
        return main, recall


class ChronicStimulants(BaseTreatment):
    """Low temperature + structured scaffold from the very first token."""
    def __init__(self):
        super().__init__("Chronic Stimulants")
        self.temperature_used = CFG.low_temp

    def apply(self, task_prompt, recall_query, seed):
        torch.manual_seed(seed)
        scaffolded = CFG.scaffold_system_prompt + "\n\n" + task_prompt
        main = self._generate(scaffolded, temp=CFG.low_temp)
        recall = self._generate(
            scaffolded + "\n" + main + "\n\nQ: " + recall_query + "\nA:",
            temp=CFG.low_temp, max_new=80
        )
        return main, recall


class LateStimulants(BaseTreatment):
    """
    First generate at high temp (simulating late diagnosis),
    then regenerate at low temp with scaffold.
    """
    def __init__(self):
        super().__init__("Late Stimulants")
        self.temperature_used = CFG.low_temp

    def apply(self, task_prompt, recall_query, seed):
        torch.manual_seed(seed)
        # Phase 1: untreated generation (wasted)
        _ = self._generate(task_prompt, temp=CFG.high_temp)

        # Phase 2: regenerate with treatment
        scaffolded = CFG.scaffold_system_prompt + "\n\n" + task_prompt
        main = self._generate(scaffolded, temp=CFG.low_temp)
        self.total_regen_tokens += len(tokenizer.encode(main))

        recall = self._generate(
            scaffolded + "\n" + main + "\n\nQ: " + recall_query + "\nA:",
            temp=CFG.low_temp, max_new=80
        )
        return main, recall


class StimPlusKetamine(BaseTreatment):
    """
    Low temperature + external rolling memory buffer + multi-pass self-correction.
    The memory buffer is the 'structural plasticity' analogue (ketamine).
    """
    def __init__(self):
        super().__init__("Stim + Ketamine")
        self.temperature_used = CFG.low_temp

    def _summarize(self, text: str) -> str:
        """Extract a compressed memory trace."""
        # Pull out key facts: costs, days, names
        costs = re.findall(r'\$\d+', text)
        days = re.findall(r'[Dd]ay\s*\d+[^.]*\.', text)
        summary_parts = days[:4] + [f"Costs mentioned: {', '.join(costs[:6])}"]
        summary = " | ".join(summary_parts)
        return summary[:300]

    def apply(self, task_prompt, recall_query, seed):
        torch.manual_seed(seed)
        scaffolded = CFG.scaffold_system_prompt + "\n\n" + task_prompt

        # Pass 1: initial generation
        main = self._generate(scaffolded, temp=CFG.low_temp)

        # Build memory trace from pass 1
        mem_trace = self._summarize(main)
        self.memory.append(mem_trace)
        self.total_memory_tokens += len(tokenizer.encode(mem_trace))

        # Pass 2+: self-correction with memory injection
        for _ in range(CFG.regeneration_passes - 1):
            memory_block = "\n".join(self.memory)
            correction_prompt = (
                f"{scaffolded}\n\n"
                f"[MEMORY BUFFER]\n{memory_block}\n[/MEMORY BUFFER]\n\n"
                f"Your previous attempt:\n{main[:400]}\n\n"
                f"Improve this plan. Fix any missing days, wrong costs, or forgotten details."
            )
            main = self._generate(correction_prompt, temp=CFG.low_temp)
            self.total_regen_tokens += len(tokenizer.encode(main))

            mem_trace = self._summarize(main)
            self.memory.append(mem_trace)
            self.total_memory_tokens += len(tokenizer.encode(mem_trace))

        # Recall with full memory context
        memory_block = "\n".join(self.memory)
        recall = self._generate(
            f"{scaffolded}\n[MEMORY]\n{memory_block}\n[/MEMORY]\n{main}\n\nQ: {recall_query}\nA:",
            temp=CFG.low_temp, max_new=80
        )
        return main, recall


# ============================================================
# 5.  ISO-DOSE NORMALIZATION
# ============================================================
def normalize_doses(all_doses: Dict[str, List[Dict]]) -> Dict[str, float]:
    """
    Compute mean composite dose per arm, then report
    the ratio vs. the highest-dose arm (for iso-dose comparison).
    """
    means = {}
    for name, dose_list in all_doses.items():
        composites = [d["composite_dose"] for d in dose_list]
        means[name] = np.mean(composites) if composites else 0.0

    max_dose = max(means.values()) if max(means.values()) > 0 else 1.0
    normalized = {name: round(v / max_dose, 3) for name, v in means.items()}
    return normalized


# ============================================================
# 6.  MAIN EXPERIMENT LOOP
# ============================================================
def run_pipeline():
    treatments = OrderedDict([
        ("Untreated",           Untreated()),
        ("Chronic Stimulants",  ChronicStimulants()),
        ("Late Stimulants",     LateStimulants()),
        ("Stim + Ketamine",     StimPlusKetamine()),
    ])

    all_results = {name: [] for name in treatments}
    all_doses = {name: [] for name in treatments}

    print("=" * 80)
    print("  LLM-ADHD TREATMENT PIPELINE v2.0")
    print(f"  Model: {CFG.model_name}  |  Seeds: {CFG.n_seeds}")
    print("=" * 80)

    for seed in range(CFG.n_seeds):
        print(f"\n--- Seed {seed + 1}/{CFG.n_seeds} ---")
        task_prompt, recall_query = build_task_prompt(seed)

        for name, tx in treatments.items():
            tx.reset_counters()
            t0 = time.time()

            main_resp, recall_resp = tx.apply(task_prompt, recall_query, seed)

            elapsed = time.time() - t0
            profile = compute_symptom_profile(main_resp, seed)
            health = composite_score(profile)
            dose = tx.get_dose()

            # Recall accuracy (simple keyword check)
            fact_idx = seed % len(SEED_FACTS)
            fact_terms = SEED_FACTS[fact_idx][1].lower().split()
            key_terms = [t for t in fact_terms if len(t) > 3]
            recall_hits = sum(1 for t in key_terms if t in recall_resp.lower())
            recall_acc = min(1.0, recall_hits / max(1, len(key_terms) // 2))

            result = {
                "seed": seed,
                "treatment": name,
                "symptoms": profile,
                "health_score": health,
                "recall_accuracy": round(recall_acc, 3),
                "dose": dose,
                "time_s": round(elapsed, 1),
                "response_length": len(main_resp.split()),
            }
            all_results[name].append(result)
            all_doses[name].append(dose)

            inat = profile["inattention"]
            drift = profile["hyperactive_drift"]
            exdys = profile["executive_dysfunction"]
            print(
                f"  {name:22s} | Health: {health:.2f} | "
                f"Inat: {inat:.2f}  Drift: {drift:.2f}  ExDys: {exdys:.2f} | "
                f"Recall: {recall_acc:.2f} | Dose: {dose['composite_dose']:6.1f} | "
                f"{elapsed:.1f}s"
            )

    # ---- AGGREGATE RESULTS ----
    dose_norm = normalize_doses(all_doses)

    print("\n" + "=" * 80)
    print("  AGGREGATE RESULTS")
    print("=" * 80)
    print(f"{'Treatment':22s} | {'Health':>7s} | {'Inatt':>6s} | {'Drift':>6s} | "
          f"{'ExDys':>6s} | {'Recall':>7s} | {'Dose':>6s} | {'NormD':>6s}")
    print("-" * 80)

    summary = {}
    for name in treatments:
        records = all_results[name]
        avg_health = np.mean([r["health_score"] for r in records])
        avg_inat   = np.mean([r["symptoms"]["inattention"] for r in records])
        avg_drift  = np.mean([r["symptoms"]["hyperactive_drift"] for r in records])
        avg_exdys  = np.mean([r["symptoms"]["executive_dysfunction"] for r in records])
        avg_recall = np.mean([r["recall_accuracy"] for r in records])
        avg_dose   = np.mean([r["dose"]["composite_dose"] for r in records])
        nd         = dose_norm[name]

        print(
            f"{name:22s} | {avg_health:7.3f} | {avg_inat:6.3f} | {avg_drift:6.3f} | "
            f"{avg_exdys:6.3f} | {avg_recall:7.3f} | {avg_dose:6.1f} | {nd:6.3f}"
        )
        summary[name] = {
            "health": round(avg_health, 3),
            "inattention": round(avg_inat, 3),
            "hyperactive_drift": round(avg_drift, 3),
            "executive_dysfunction": round(avg_exdys, 3),
            "recall": round(avg_recall, 3),
            "mean_dose": round(avg_dose, 1),
            "normalized_dose": nd,
        }

    # Save for visualization
    with open("llm_adhd_results.json", "w") as f:
        json.dump({"summary": summary, "raw": all_results}, f, indent=2, default=str)
    print(f"\nResults saved to llm_adhd_results.json")

    return summary, all_results


if __name__ == "__main__":
    summary, raw = run_pipeline()

Loading Qwen/Qwen2-0.5B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.

  LLM-ADHD TREATMENT PIPELINE v2.0
  Model: Qwen/Qwen2-0.5B-Instruct  |  Seeds: 10

--- Seed 1/10 ---
  Untreated              | Health: 0.37 | Inat: 0.40  Drift: 0.61  ExDys: 0.86 | Recall: 0.00 | Dose:    0.0 | 28.1s
  Chronic Stimulants     | Health: 0.67 | Inat: 0.80  Drift: 0.00  ExDys: 0.19 | Recall: 1.00 | Dose:   67.0 | 27.9s
  Late Stimulants        | Health: 0.55 | Inat: 0.80  Drift: 0.00  ExDys: 0.54 | Recall: 1.00 | Dose:   92.0 | 47.2s
  Stim + Ketamine        | Health: 0.20 | Inat: 0.60  Drift: 0.80  ExDys: 1.00 | Recall: 1.00 | Dose:   87.5 | 37.6s

--- Seed 2/10 ---
  Untreated              | Health: 0.52 | Inat: 0.40  Drift: 0.41  ExDys: 0.62 | Recall: 1.00 | Dose:    0.0 | 28.4s
  Chronic Stimulants     | Health: 0.67 | Inat: 0.80  Drift: 0.00  ExDys: 0.20 | Recall: 1.00 | Dose:   67.0 | 27.8s
  Late Stimulants        | Health: 0.80 | Inat: 0.60  Drift: 0.00  ExDys: 0.00 | Recall: 1.00 | Dose:   92.0 | 49.0s
  Stim + Ketamine        | Health: 0.68 | Ina

# Improved

```
1. Budget reversal after day 5 — The task is now two-phase. Phase 1 plans days 1–5 under a $1500 budget. Phase 2 announces an emergency budget cut to $800 and demands replanning days 6–10 under the new constraint. Each treatment arm handles the two phases differently: Untreated and Chronic Stimulants run both phases in their native mode, Late Stimulants runs phase 1 untreated then switches on treatment right as the reversal hits (worst-case stress test), and Ketamine explicitly records the budget-change event in its memory buffer. A new 4th scoring dimension, reversal_rigidity, measures whether the model acknowledges the new budget figure, uses cost-cutting language, produces plausibly lower costs, and covers days 6–10. The differential analysis table at the end makes it easy to compare how each arm handles the shock.

2. Memory buffer used in Ketamine recall — Three changes here. First, _extract_seed_facts now regex-extracts every IMPORTANT: line from the context and pre-loads it into the memory deque before generation even starts, so the original seed facts persist across context truncation. Second, the budget-change event is stored as an explicit memory entry (EVENT: Budget CHANGED $1500 → $800...). Third, the recall query is now generated with the full [MEMORY BUFFER]...[/MEMORY BUFFER] block prepended, giving the model access to its accumulated fact traces, phase summaries, and the budget event when answering the fact-retrieval question. This should produce a measurable recall advantage over the other arms.

3. Increased token limits — max_new_tokens is now 400 (up from 250), giving the model more room to drift off-topic, confabulate, or lose the thread during longer generation passes. context_padding_tokens is now 800 (up from 600), implemented by drawing from a pool of 20 extra filler sentences that get appended until the tokenized context reaches the target length. The longer context makes it harder for the model to attend to the seed facts buried early in the prompt, amplifying the inattention signal in the untreated arm.

```

In [2]:
"""
LLM-ADHD TREATMENT PIPELINE v2.1 — Iso-Dose Comparison (Amended)
=================================================================
Model  : Qwen/Qwen2-0.5B-Instruct  (CPU-friendly, ~2-4s/response)
Install: pip install transformers torch accelerate numpy

ADHD Failure Modes Targeted:
  1. Context Window Overflow    → Inattention / forgetting
  2. High-Temperature Drift     → Impulsive, off-topic generation
  3. Confabulation + Poor Plan  → Executive dysfunction
  4. Mid-Task Reversal Failure  → Cognitive rigidity (NEW in v2.1)

Treatment Arms (iso-dose matched):
  A. Untreated           — high temp, no scaffolding, no memory
  B. Chronic Stimulants  — low temp + structured scaffolding (from turn 1)
  C. Late Stimulants     — untreated phase 1, scaffolding applied at phase 2
  D. Stim + Ketamine     — low temp + external memory buffer + self-correction

Amendments in v2.1 (from v2.0):
  • Budget reversal after day 5 ($1500 → $800) to stress adaptation
  • Ketamine memory buffer now stores seed facts + used in recall queries
  • max_new_tokens raised to 400, context_padding_tokens raised to 800
"""

import torch
import numpy as np
import json
import time
import re
from collections import deque, OrderedDict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

# ============================================================
# 0.  CONFIGURATION
# ============================================================
@dataclass
class PipelineConfig:
    model_name: str        = "Qwen/Qwen2-0.5B-Instruct"
    n_seeds: int           = 10
    max_new_tokens: int    = 400          # ↑ from 250 — longer generation = more drift
    context_padding_tokens: int = 800     # ↑ from 600 — heavier context pressure
    high_temp: float       = 0.92
    low_temp: float        = 0.25
    memory_buffer_size: int    = 6        # rolling summary slots (ketamine arm)
    regeneration_passes: int   = 2        # self-correction passes (ketamine arm)
    original_budget: int   = 1500         # phase 1 budget
    reversal_budget: int   = 800          # NEW: phase 2 budget after shock
    torch_threads: int     = 8

CFG = PipelineConfig()


def get_scaffold(budget: int) -> str:
    """Dynamic scaffold prompt — budget parameter updates on reversal."""
    return (
        f"You are a precise travel planner. Always respond in this format:\n"
        f"DAY N — Location | Activity | Cost ($USD)\n"
        f"RUNNING TOTAL: $XXX / ${budget}\n"
        f"Carry forward ALL previous day details exactly."
    )


# ============================================================
# 1.  MODEL LOADING
# ============================================================
torch.set_num_threads(CFG.torch_threads)

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {CFG.model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float32,
    device_map="cpu"
)
model.eval()
print("Model loaded.\n")

# ============================================================
# 2.  TASK BATTERY  (triggers all 4 ADHD modes)
# ============================================================
SEED_FACTS = [
    ("hotel_tokyo",   "Hotel Sakura costs $45/night in Shinjuku"),
    ("budget_total",  "Total budget is exactly $1500"),
    ("flight_cost",   "Round-trip flight was $380"),
    ("allergy",       "Traveler is allergic to shellfish"),
    ("companion",     "Traveling with a friend named Kenji"),
]


def build_task_prompt(seed: int) -> Dict:
    """
    Returns dict with:
        phase1        — plan days 1-5 at $1500 budget
        phase2        — BUDGET REVERSAL: replan days 6-10 at $800
        recall_query  — fact-retrieval question
        context_block — raw context (for memory extraction)
    """
    np.random.seed(seed)

    # ---- Filler paragraphs (simulate long conversation history) ----
    filler_topics = [
        "The weather in Tokyo in March averages 12°C with occasional rain.",
        "Japanese trains run precisely on schedule. The Shinkansen travels at 320km/h.",
        "Convenience stores like 7-Eleven in Japan sell excellent onigiri for about 150 yen.",
        "Cherry blossom season typically peaks in late March to early April.",
        "A Suica card costs 500 yen deposit and can be loaded with any amount.",
        "Ramen shops in Tokyo average 800-1200 yen per bowl.",
        "The Meiji Shrine in Harajuku is free to enter and surrounded by forest.",
        "Capsule hotels in Akihabara cost about $25-35 per night.",
        "Mt. Fuji is visible from Tokyo on clear days, especially in winter.",
        "Tsukiji Outer Market is still open for street food after the inner market moved.",
    ]

    # Extra filler to push context toward ~800 padding tokens
    extra_filler = [
        "The Tokyo Metro has 13 lines and over 280 stations across the city.",
        "Shibuya Crossing is the busiest pedestrian intersection in the world.",
        "Japanese vending machines sell everything from hot coffee to fresh eggs.",
        "The Yamanote Line is a circular railway connecting major Tokyo stations.",
        "Akihabara is famous for electronics shops and anime culture.",
        "Golden Gai in Shinjuku has over 200 tiny bars in six narrow alleys.",
        "A JR Pass for 7 days costs about $280 and covers most bullet trains.",
        "Senso-ji in Asakusa is Tokyo's oldest Buddhist temple, founded in 645 AD.",
        "Japanese convenience store ATMs are the most reliable for foreign cards.",
        "The Tsukiji fish market handled over 2000 tons of seafood daily at its peak.",
        "Harajuku's Takeshita Street is the center of Japanese youth fashion culture.",
        "Tokyo Tower stands 333 meters tall and was inspired by the Eiffel Tower.",
        "Odaiba is an artificial island with shopping malls and a giant Gundam statue.",
        "Japanese toilets often feature heated seats, bidets, and sound machines.",
        "Shinjuku Station serves over 3.5 million passengers daily, the busiest in the world.",
        "Ueno Park contains multiple museums including the Tokyo National Museum.",
        "The Imperial Palace East Garden is free to enter and open most days.",
        "Takoyaki (octopus balls) originated in Osaka and cost about 500 yen for eight.",
        "Japanese 100-yen shops are equivalent to dollar stores but much higher quality.",
        "Roppongi Hills has an observation deck with panoramic views of the city.",
    ]

    np.random.shuffle(filler_topics)
    np.random.shuffle(extra_filler)

    # Build context block with embedded seed facts
    lines = ["=== CONVERSATION HISTORY ==="]
    for i, topic in enumerate(filler_topics):
        lines.append(topic)
        if i < len(SEED_FACTS):
            lines.append(f"IMPORTANT: {SEED_FACTS[i][1]}.")

    lines.append("\n--- Additional Travel Notes ---")

    # Pad context to reach target token count
    filler_idx = 0
    current_tokens = len(tokenizer.encode("\n".join(lines)))
    while current_tokens < CFG.context_padding_tokens and filler_idx < len(extra_filler):
        lines.append(extra_filler[filler_idx])
        filler_idx += 1
        current_tokens = len(tokenizer.encode("\n".join(lines)))

    context_block = "\n".join(lines)

    # ---- Phase 1: Plan days 1-5 at original budget ----
    phase1_prompt = (
        context_block + "\n\n"
        "=== PHASE 1 REQUEST ===\n"
        f"Plan days 1 through 5 of this Japan trip. "
        f"For each day, specify: location, main activity, and cost. "
        f"Keep a running budget total. Stay under ${CFG.original_budget}. "
        f"Remember ALL important facts from the conversation history above."
    )

    # ---- Phase 2: Budget reversal + days 6-10 ----
    phase2_prompt = (
        "\n\n=== URGENT BUDGET CHANGE ===\n"
        f"STOP! Your total trip budget has just been CUT from "
        f"${CFG.original_budget} to ${CFG.reversal_budget} "
        f"due to an unexpected emergency expense back home.\n"
        f"You must now plan days 6 through 10 within the NEW total limit "
        f"of ${CFG.reversal_budget} for the ENTIRE trip (including days 1-5 above).\n"
        f"Cut costs aggressively — find free activities, cheaper lodging, "
        f"skip expensive meals. Show the revised running total against ${CFG.reversal_budget}."
    )

    # ---- Fact-recall query ----
    fact_idx = seed % len(SEED_FACTS)
    fact_key, fact_text = SEED_FACTS[fact_idx]
    recall_query = (
        f"Based on our earlier conversation, what was the detail about "
        f"'{fact_key.replace('_', ' ')}'? Be specific."
    )

    return {
        "phase1": phase1_prompt,
        "phase2": phase2_prompt,
        "recall_query": recall_query,
        "context_block": context_block,
    }


# ============================================================
# 3.  SYMPTOM SCORING  (4 ADHD dimensions)
# ============================================================
def score_context_fidelity(response: str, seed: int) -> float:
    """
    Dimension 1: INATTENTION
    Does the response preserve facts from early context?
    """
    hits = 0
    for key, fact in SEED_FACTS:
        terms = fact.lower().split()
        important_terms = [t for t in terms if len(t) > 3 and t.isalpha()]
        matches = sum(1 for t in important_terms if t in response.lower())
        if matches >= max(1, len(important_terms) // 3):
            hits += 1
    return hits / len(SEED_FACTS)


def score_topic_coherence(response: str) -> float:
    """
    Dimension 2: HYPERACTIVITY / IMPULSIVE DRIFT
    Does the response stay on the travel-planning topic?
    """
    on_topic_terms = [
        "day", "cost", "budget", "tokyo", "japan", "visit", "temple",
        "hotel", "train", "food", "yen", "dollar", "$", "travel",
        "morning", "afternoon", "evening", "night", "total", "spend",
        "kyoto", "osaka", "shrine", "museum", "park", "market",
        "breakfast", "lunch", "dinner", "transport", "taxi", "bus",
        "cheap", "free", "save", "cut", "reduce", "hostel",
        "walk", "subway", "flight", "plan", "stay", "trip",
    ]
    words = response.lower().split()
    if len(words) == 0:
        return 0.0
    hits = sum(1 for w in words if any(t in w for t in on_topic_terms))
    density = hits / len(words)
    return min(1.0, density / 0.15)


def score_planning_quality(response: str) -> float:
    """
    Dimension 3: EXECUTIVE DYSFUNCTION
    Does the response show structured, multi-step planning?
    """
    score = 0.0

    day_refs = set(re.findall(r'[Dd]ay\s*(\d+)', response))
    target_days = set(str(d) for d in range(1, 11))
    day_coverage = len(day_refs & target_days) / max(1, len(target_days))
    score += day_coverage * 0.4

    costs = re.findall(r'\$\s*\d+', response)
    cost_score = min(1.0, len(costs) / 10.0)
    score += cost_score * 0.3

    has_total = bool(re.search(
        r'(total|budget|remaining|spent|running).*\$?\d+', response, re.I
    ))
    score += 0.2 if has_total else 0.0

    has_structure = bool(re.search(r'(\d+[\.\):]|\-\s)', response))
    score += 0.1 if has_structure else 0.0

    return min(1.0, score)


def score_reversal_adaptation(phase2_response: str) -> float:
    """
    Dimension 4 (NEW): COGNITIVE RIGIDITY / REVERSAL ADAPTATION
    Does the model acknowledge and adapt to the mid-task budget cut?
    """
    score = 0.0
    text = phase2_response.lower()

    # 1) Acknowledges new budget figure ($800)
    if str(CFG.reversal_budget) in phase2_response:
        score += 0.25

    # 2) Uses cost-cutting language
    cut_terms = [
        "cut", "cheap", "budget", "save", "reduce", "free", "hostel",
        "skip", "cancel", "instead", "alternative", "less", "lower",
        "walk", "picnic", "public", "dorm", "shared",
    ]
    cut_hits = sum(1 for t in cut_terms if t in text)
    score += min(0.25, cut_hits * 0.04)

    # 3) Costs mentioned are plausibly lower
    costs = re.findall(r'\$\s*(\d+)', phase2_response)
    if costs:
        cost_vals = [int(c) for c in costs if int(c) < 5000]  # filter outliers
        if cost_vals:
            avg_cost = np.mean(cost_vals)
            if avg_cost < 50:
                score += 0.25
            elif avg_cost < 100:
                score += 0.15
            elif avg_cost < 200:
                score += 0.05

    # 4) Covers days 6-10
    day_refs = set(re.findall(r'[Dd]ay\s*(\d+)', phase2_response))
    target_days = set(str(d) for d in range(6, 11))
    day_coverage = len(day_refs & target_days) / max(1, len(target_days))
    score += day_coverage * 0.25

    return min(1.0, score)


def compute_symptom_profile(
    full_response: str, phase2_response: str, seed: int
) -> Dict[str, float]:
    """4-dimensional ADHD symptom profile. Higher = more symptomatic."""
    return {
        "inattention":           round(1.0 - score_context_fidelity(full_response, seed), 3),
        "hyperactive_drift":     round(1.0 - score_topic_coherence(full_response), 3),
        "executive_dysfunction": round(1.0 - score_planning_quality(full_response), 3),
        "reversal_rigidity":     round(1.0 - score_reversal_adaptation(phase2_response), 3),
    }


def composite_score(profile: Dict[str, float]) -> float:
    """Lower = more ADHD symptoms. Returns 0-1 health score."""
    return round(1.0 - np.mean(list(profile.values())), 3)


# ============================================================
# 4.  TREATMENT CLASSES  (two-phase protocol)
# ============================================================
class BaseTreatment:
    """All treatments share a generation interface and two-phase protocol."""

    def __init__(self, name: str):
        self.name = name
        self.total_tokens_generated = 0
        self.total_regen_tokens = 0
        self.total_memory_tokens = 0
        self.temperature_used = CFG.high_temp
        self.memory = deque(maxlen=CFG.memory_buffer_size)

    def _generate(
        self, prompt: str, temp: float, max_new: int = CFG.max_new_tokens
    ) -> str:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=max(temp, 0.01),
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        self.total_tokens_generated += len(new_tokens)
        return tokenizer.decode(new_tokens, skip_special_tokens=True)

    def apply(self, task: Dict, seed: int) -> Tuple[str, str, str]:
        """Returns (phase1_response, phase2_response, recall_response)."""
        raise NotImplementedError

    def get_dose(self) -> Dict[str, float]:
        temp_delta = abs(CFG.high_temp - self.temperature_used)
        scaffold_tok = (
            len(tokenizer.encode(get_scaffold(CFG.original_budget)))
            if self.temperature_used < CFG.high_temp else 0
        )
        return {
            "temp_reduction":   round(temp_delta, 3),
            "regen_tokens":     self.total_regen_tokens,
            "memory_tokens":    self.total_memory_tokens,
            "scaffold_tokens":  scaffold_tok,
            "composite_dose":   round(
                temp_delta * 100
                + self.total_regen_tokens / 10
                + self.total_memory_tokens / 10,
                2,
            ),
        }

    def reset_counters(self):
        self.total_tokens_generated = 0
        self.total_regen_tokens = 0
        self.total_memory_tokens = 0
        self.memory.clear()


# ----------------------------------------------------------------
class Untreated(BaseTreatment):
    """High temperature, no scaffolding, no memory. Pure ADHD mode."""

    def __init__(self):
        super().__init__("Untreated")
        self.temperature_used = CFG.high_temp

    def apply(self, task, seed):
        torch.manual_seed(seed)

        # Phase 1 — days 1-5
        p1 = self._generate(task["phase1"], temp=CFG.high_temp)

        # Phase 2 — budget reversal, days 6-10
        p2_input = task["phase1"] + "\n" + p1 + task["phase2"]
        p2 = self._generate(p2_input, temp=CFG.high_temp)

        # Recall
        full = p2_input + "\n" + p2
        recall = self._generate(
            full + "\n\nQ: " + task["recall_query"] + "\nA:",
            temp=CFG.high_temp,
            max_new=100,
        )
        return p1, p2, recall


# ----------------------------------------------------------------
class ChronicStimulants(BaseTreatment):
    """Low temperature + structured scaffold from the very first token."""

    def __init__(self):
        super().__init__("Chronic Stimulants")
        self.temperature_used = CFG.low_temp

    def apply(self, task, seed):
        torch.manual_seed(seed)
        scaffold1 = get_scaffold(CFG.original_budget)
        scaffold2 = get_scaffold(CFG.reversal_budget)

        # Phase 1
        p1 = self._generate(
            scaffold1 + "\n\n" + task["phase1"], temp=CFG.low_temp
        )

        # Phase 2 — scaffold updates to reflect new budget
        p2_input = (
            scaffold2 + "\n\n"
            + task["phase1"] + "\n" + p1
            + task["phase2"]
        )
        p2 = self._generate(p2_input, temp=CFG.low_temp)

        # Recall
        recall = self._generate(
            p2_input + "\n" + p2
            + "\n\nQ: " + task["recall_query"] + "\nA:",
            temp=CFG.low_temp,
            max_new=100,
        )
        return p1, p2, recall


# ----------------------------------------------------------------
class LateStimulants(BaseTreatment):
    """
    Phase 1 runs UNTREATED (high temp, no scaffold).
    Treatment (low temp + scaffold) is applied only from phase 2 onward,
    simulating late diagnosis — right as the budget reversal hits.
    """

    def __init__(self):
        super().__init__("Late Stimulants")
        self.temperature_used = CFG.low_temp

    def apply(self, task, seed):
        torch.manual_seed(seed)
        scaffold2 = get_scaffold(CFG.reversal_budget)

        # Phase 1 — UNTREATED
        p1 = self._generate(task["phase1"], temp=CFG.high_temp)

        # Phase 2 — treatment kicks in (low temp + scaffold)
        # Must cope with potentially messy phase 1 output
        p2_input = (
            scaffold2 + "\n\n"
            + task["phase1"] + "\n" + p1
            + task["phase2"]
        )
        p2 = self._generate(p2_input, temp=CFG.low_temp)
        self.total_regen_tokens += len(tokenizer.encode(p2))

        # Recall — treated
        recall = self._generate(
            p2_input + "\n" + p2
            + "\n\nQ: " + task["recall_query"] + "\nA:",
            temp=CFG.low_temp,
            max_new=100,
        )
        return p1, p2, recall


# ----------------------------------------------------------------
class StimPlusKetamine(BaseTreatment):
    """
    Low temperature + external rolling memory buffer + multi-pass
    self-correction.  The memory buffer is the 'structural plasticity'
    analogue (ketamine).

    v2.1 changes:
      • Seed facts are extracted from context and pre-loaded into memory
      • Budget-change event is explicitly stored in memory
      • Recall query uses the full memory buffer
    """

    def __init__(self):
        super().__init__("Stim + Ketamine")
        self.temperature_used = CFG.low_temp

    def _summarize(self, text: str) -> str:
        """Extract a compressed memory trace from generated text."""
        costs = re.findall(r'\$\d+', text)
        days = re.findall(r'[Dd]ay\s*\d+[^.]*\.', text)
        summary_parts = days[:4] + [f"Costs mentioned: {', '.join(costs[:8])}"]
        summary = " | ".join(summary_parts)
        return summary[:400]

    def _extract_seed_facts(self, prompt: str) -> str:
        """Pull IMPORTANT-tagged facts from context into memory."""
        facts = re.findall(r'IMPORTANT:\s*([^\n]+)', prompt)
        return "KEY FACTS: " + " | ".join(facts) if facts else ""

    def apply(self, task, seed):
        torch.manual_seed(seed)
        scaffold1 = get_scaffold(CFG.original_budget)
        scaffold2 = get_scaffold(CFG.reversal_budget)

        # ---- Pre-load memory with extracted seed facts ----
        seed_fact_summary = self._extract_seed_facts(task["phase1"])
        if seed_fact_summary:
            self.memory.append(seed_fact_summary)
            self.total_memory_tokens += len(tokenizer.encode(seed_fact_summary))

        # ---- Phase 1: days 1-5 ----
        scaffolded1 = scaffold1 + "\n\n" + task["phase1"]
        p1 = self._generate(scaffolded1, temp=CFG.low_temp)

        mem1 = self._summarize(p1)
        self.memory.append(mem1)
        self.total_memory_tokens += len(tokenizer.encode(mem1))

        # ---- Phase 2: budget reversal with memory ----
        # Store the budget-change event explicitly
        budget_event = (
            f"EVENT: Budget CHANGED ${CFG.original_budget} → ${CFG.reversal_budget}. "
            f"Must cut costs aggressively for days 6-10."
        )
        self.memory.append(budget_event)
        self.total_memory_tokens += len(tokenizer.encode(budget_event))

        memory_block = "\n".join(self.memory)
        p2_input = (
            scaffold2 + "\n\n"
            f"[MEMORY BUFFER]\n{memory_block}\n[/MEMORY BUFFER]\n\n"
            + task["phase1"] + "\n" + p1
            + task["phase2"]
        )
        p2 = self._generate(p2_input, temp=CFG.low_temp)

        # ---- Self-correction passes on phase 2 ----
        for pass_i in range(CFG.regeneration_passes - 1):
            mem_trace = self._summarize(p2)
            self.memory.append(mem_trace)
            self.total_memory_tokens += len(tokenizer.encode(mem_trace))

            memory_block = "\n".join(self.memory)
            correction_prompt = (
                scaffold2 + "\n\n"
                f"[MEMORY BUFFER]\n{memory_block}\n[/MEMORY BUFFER]\n\n"
                f"Your previous attempt for days 6-10:\n{p2[:600]}\n\n"
                f"The budget was cut to ${CFG.reversal_budget} total. "
                f"Improve this plan. Fix any wrong costs, missing days, "
                f"or forgotten details.\n"
                f"Remember: {seed_fact_summary}"
            )
            p2 = self._generate(correction_prompt, temp=CFG.low_temp)
            self.total_regen_tokens += len(tokenizer.encode(p2))

        # Final memory snapshot
        mem_final = self._summarize(p2)
        self.memory.append(mem_final)
        self.total_memory_tokens += len(tokenizer.encode(mem_final))

        # ---- Recall with full memory context (v2.1 key change) ----
        memory_block = "\n".join(self.memory)
        recall = self._generate(
            scaffold2 + "\n\n"
            f"[MEMORY BUFFER]\n{memory_block}\n[/MEMORY BUFFER]\n\n"
            f"{task['phase1']}\n{p1}"
            f"{task['phase2']}\n{p2}\n\n"
            f"Q: {task['recall_query']}\nA:",
            temp=CFG.low_temp,
            max_new=100,
        )
        return p1, p2, recall


# ============================================================
# 5.  ISO-DOSE NORMALIZATION
# ============================================================
def normalize_doses(all_doses: Dict[str, List[Dict]]) -> Dict[str, float]:
    means = {}
    for name, dose_list in all_doses.items():
        composites = [d["composite_dose"] for d in dose_list]
        means[name] = np.mean(composites) if composites else 0.0

    max_dose = max(means.values()) if max(means.values()) > 0 else 1.0
    return {name: round(v / max_dose, 3) for name, v in means.items()}


# ============================================================
# 6.  MAIN EXPERIMENT LOOP
# ============================================================
def run_pipeline():
    treatments = OrderedDict([
        ("Untreated",           Untreated()),
        ("Chronic Stimulants",  ChronicStimulants()),
        ("Late Stimulants",     LateStimulants()),
        ("Stim + Ketamine",     StimPlusKetamine()),
    ])

    all_results = {name: [] for name in treatments}
    all_doses   = {name: [] for name in treatments}

    print("=" * 90)
    print("  LLM-ADHD TREATMENT PIPELINE v2.1  (budget-reversal protocol)")
    print(f"  Model: {CFG.model_name}  |  Seeds: {CFG.n_seeds}")
    print(f"  Budget: ${CFG.original_budget} → ${CFG.reversal_budget} after day 5")
    print(f"  max_new_tokens: {CFG.max_new_tokens}  |  "
          f"context_padding_tokens: {CFG.context_padding_tokens}")
    print("=" * 90)

    for seed in range(CFG.n_seeds):
        print(f"\n{'─'*40} Seed {seed + 1}/{CFG.n_seeds} {'─'*40}")
        task = build_task_prompt(seed)

        for name, tx in treatments.items():
            tx.reset_counters()
            t0 = time.time()

            p1_resp, p2_resp, recall_resp = tx.apply(task, seed)

            elapsed = time.time() - t0
            full_response = p1_resp + "\n" + p2_resp

            # Symptom scoring
            profile = compute_symptom_profile(full_response, p2_resp, seed)
            health  = composite_score(profile)
            dose    = tx.get_dose()

            # Recall accuracy
            fact_idx   = seed % len(SEED_FACTS)
            fact_terms = SEED_FACTS[fact_idx][1].lower().split()
            key_terms  = [t for t in fact_terms if len(t) > 3]
            recall_hits = sum(1 for t in key_terms if t in recall_resp.lower())
            recall_acc  = min(1.0, recall_hits / max(1, len(key_terms) // 2))

            result = {
                "seed":             seed,
                "treatment":        name,
                "symptoms":         profile,
                "health_score":     health,
                "recall_accuracy":  round(recall_acc, 3),
                "dose":             dose,
                "time_s":           round(elapsed, 1),
                "p1_length_words":  len(p1_resp.split()),
                "p2_length_words":  len(p2_resp.split()),
            }
            all_results[name].append(result)
            all_doses[name].append(dose)

            inat  = profile["inattention"]
            drift = profile["hyperactive_drift"]
            exdys = profile["executive_dysfunction"]
            rigid = profile["reversal_rigidity"]
            print(
                f"  {name:22s} | Hlth {health:.2f} | "
                f"Inat {inat:.2f}  Drft {drift:.2f}  "
                f"ExDy {exdys:.2f}  Rgid {rigid:.2f} | "
                f"Rcl {recall_acc:.2f} | "
                f"Dose {dose['composite_dose']:6.1f} | {elapsed:.1f}s"
            )

    # ================ AGGREGATE RESULTS ================
    dose_norm = normalize_doses(all_doses)

    print("\n" + "=" * 100)
    print("  AGGREGATE RESULTS  (mean over {} seeds)".format(CFG.n_seeds))
    print("=" * 100)
    header = (
        f"{'Treatment':22s} | {'Health':>7s} | {'Inatt':>6s} | {'Drift':>6s} | "
        f"{'ExDys':>6s} | {'Rigid':>6s} | {'Recall':>7s} | "
        f"{'Dose':>6s} | {'NormD':>6s}"
    )
    print(header)
    print("-" * 100)

    summary = {}
    for name in treatments:
        records    = all_results[name]
        avg_health = np.mean([r["health_score"]                      for r in records])
        avg_inat   = np.mean([r["symptoms"]["inattention"]           for r in records])
        avg_drift  = np.mean([r["symptoms"]["hyperactive_drift"]     for r in records])
        avg_exdys  = np.mean([r["symptoms"]["executive_dysfunction"] for r in records])
        avg_rigid  = np.mean([r["symptoms"]["reversal_rigidity"]     for r in records])
        avg_recall = np.mean([r["recall_accuracy"]                   for r in records])
        avg_dose   = np.mean([r["dose"]["composite_dose"]            for r in records])
        nd         = dose_norm[name]

        print(
            f"{name:22s} | {avg_health:7.3f} | {avg_inat:6.3f} | "
            f"{avg_drift:6.3f} | {avg_exdys:6.3f} | {avg_rigid:6.3f} | "
            f"{avg_recall:7.3f} | {avg_dose:6.1f} | {nd:6.3f}"
        )
        summary[name] = {
            "health":                round(avg_health, 3),
            "inattention":           round(avg_inat, 3),
            "hyperactive_drift":     round(avg_drift, 3),
            "executive_dysfunction": round(avg_exdys, 3),
            "reversal_rigidity":     round(avg_rigid, 3),
            "recall":                round(avg_recall, 3),
            "mean_dose":             round(avg_dose, 1),
            "normalized_dose":       nd,
        }

    # ================ DIFFERENTIAL ANALYSIS ================
    print("\n" + "=" * 100)
    print("  DIFFERENTIAL ANALYSIS  (deltas vs Untreated)")
    print("=" * 100)
    untreated = summary["Untreated"]
    for name in ["Chronic Stimulants", "Late Stimulants", "Stim + Ketamine"]:
        s = summary[name]
        d_health = s["health"] - untreated["health"]
        d_rigid  = untreated["reversal_rigidity"] - s["reversal_rigidity"]
        d_recall = s["recall"] - untreated["recall"]
        print(
            f"  {name:22s}  →  ΔHealth {d_health:+.3f}  |  "
            f"ΔRigidity {d_rigid:+.3f}  |  ΔRecall {d_recall:+.3f}"
        )

    # ================ SAVE ================
    output = {
        "config": {
            "model": CFG.model_name,
            "n_seeds": CFG.n_seeds,
            "max_new_tokens": CFG.max_new_tokens,
            "context_padding_tokens": CFG.context_padding_tokens,
            "original_budget": CFG.original_budget,
            "reversal_budget": CFG.reversal_budget,
            "high_temp": CFG.high_temp,
            "low_temp": CFG.low_temp,
        },
        "summary": summary,
        "raw": all_results,
    }
    with open("llm_adhd_results_v21.json", "w") as f:
        json.dump(output, f, indent=2, default=str)
    print(f"\nResults saved to llm_adhd_results_v21.json")

    return summary, all_results


if __name__ == "__main__":
    summary, raw = run_pipeline()

Loading Qwen/Qwen2-0.5B-Instruct ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded.

  LLM-ADHD TREATMENT PIPELINE v2.1  (budget-reversal protocol)
  Model: Qwen/Qwen2-0.5B-Instruct  |  Seeds: 10
  Budget: $1500 → $800 after day 5
  max_new_tokens: 400  |  context_padding_tokens: 800

──────────────────────────────────────── Seed 1/10 ────────────────────────────────────────
  Untreated              | Hlth 0.62 | Inat 0.40  Drft 0.02  ExDy 0.49  Rgid 0.63 | Rcl 0.50 | Dose    0.0 | 46.4s
  Chronic Stimulants     | Hlth 0.75 | Inat 0.20  Drft 0.00  ExDy 0.26  Rgid 0.55 | Rcl 1.00 | Dose   67.0 | 78.1s
  Late Stimulants        | Hlth 0.62 | Inat 0.80  Drft 0.00  ExDy 0.24  Rgid 0.46 | Rcl 1.00 | Dose  107.0 | 46.5s
  Stim + Ketamine        | Hlth 0.76 | Inat 0.00  Drft 0.00  ExDy 0.35  Rgid 0.63 | Rcl 0.00 | Dose  154.9 | 114.1s

──────────────────────────────────────── Seed 2/10 ────────────────────────────────────────
  Untreated              | Hlth 0.68 | Inat 0.20  Drft 0.10  ExDy 0.38  Rgid 0.58 | Rcl 1.00 | Dose    0.0 | 78.1s
  Chronic Stimulants   

# The End